In [ ]:
import sys
import os
from pathlib import Path
# Go up until you find the project root
while not (Path.cwd() / ".gitignore").exists() and Path.cwd() != Path("/"):
    os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

: 

In [ ]:

results_df_detailed = pd.read_csv('output/integration_geoplant/france_sparse_1_0.2/partition_results_detailed_environmental.csv')
# split_types = ['closest', 'middle', 'farthest']
auc_cols   = ["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA",
               "AUC_PO_plus_PA_ens", "AUC_PO_plus_PA_smooth", "AUC_PO_plus_PA_bias"]
# auc_cols_sp = [f'{c}_sp' for c in auc_cols]
# results_df_detailed.groupby(['species', 'split_type'])[auc_cols_sp].mean().groupby('split_type').mean()
# for split_type in split_types:
#     print(f"=== Split type: {split_type} ===")
#     for col in auc_cols_sp:
#         mean_auc = results_df_detailed[results_df_detailed['split_type'] == split_type][col].mean()
#         std_auc = results_df_detailed[results_df_detailed['split_type'] == split_type][col].std()
#         print(f"{col}: {mean_auc:.4f} ± {std_auc:.4f}")
        

In [ ]:
results_df_detailed['test_id'] = results_df_detailed['split_id'] //2

In [ ]:
import numpy as np

# --- Inputs / config ---
auc_cols = [
    "AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA", "AUC_PO_plus_PA_bias"
]
auc_cols_sp = [f"{c}_sp" for c in auc_cols]
split_types = ["closest", "middle", "farthest"]

# --- Compute: per split_type, collect species-averaged distributions for each method ---
plot_data = {}  # split_type -> dict with data, means
for split in split_types:
    df_sub = results_df_detailed[results_df_detailed["split_type"] == split]

    data = []
    for c in auc_cols_sp:
        species_means = df_sub.groupby("test_id")[c].mean().values
        data.append(species_means)

    plot_data[split] = {
        "data": data,
        "means": [float(np.mean(d)) for d in data],
    }

n_box = len(auc_cols)
positions = np.linspace(-1, 1, n_box)
width = 2 / n_box


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_auc_boxplots(plot_data, auc_cols, split_types, positions, width,
                      dark_mode=False, name_map=None):
    name_map = name_map or {}
    labels = [name_map.get(c, c) for c in auc_cols]
    colors = plt.cm.tab10.colors[:len(auc_cols)]

    # optional dark mode (minimal)
    if dark_mode:
        plt.style.use("dark_background")
    else:
        plt.style.use("default")

    fig, axes = plt.subplots(1, len(split_types), figsize=(4*len(split_types), 5), sharey=True)

    if len(split_types) == 1:
        axes = [axes]

    for ax, split in zip(axes, split_types):
        data = plot_data[split]["data"]
        means = plot_data[split]["means"]

        bp = ax.boxplot(
            data, positions=positions, widths=width,
            patch_artist=True, showfliers=False,
            medianprops=dict(color="black" if not dark_mode else "white")
        )
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)

        # mean markers (kept), but not in legend
        ax.scatter(positions, means, s=50, marker="o",
                   edgecolor="black", facecolor="white", zorder=3)

        ax.set_title(split)
        ax.set_xlim(positions.min() - 0.6, positions.max() + 0.6)
        ax.set_xticks([])

    axes[0].set_ylabel("AUC")

    # add line at 0.5
    for ax in axes:
        ax.axhline(0.5, color="gray", linestyle="--", linewidth=2)

    # legend on the left, no mean entry
    handles = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, labels)]
    fig.legend(handles=handles, loc="center right", bbox_to_anchor=(0.5, -0.1), frameon=False)

    plt.tight_layout(rect=(0, 0, 0.82, 1))  # leave room for right legend
    plt.show()


# --- Use it ---
dark_mode = True

# Optional: rename methods (keys are original auc_cols)
name_map = {
    "AUC_PO_only": "PO",
    "AUC_PA_only": "PA",
    "AUC_PO_plus_PA": "PO+PA",
    "AUC_PO_plus_PA_bias": "PO+PA\n(accounting bias)"
}

plot_auc_boxplots(plot_data, auc_cols, split_types, positions, width,
                  dark_mode=dark_mode, name_map=name_map)


In [ ]:
# unique_regions = results_df['region'].unique()
# for region in unique_regions:
#     df_region = results_df[results_df['region'] == region]

#     auc_cols    = ["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA",
#                 "AUC_PO_plus_PA_ens", "AUC_PO_plus_PA_smooth"]
#     colors      = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
#     split_types = ['closest', 'middle', 'farthest']

#     fig, axes = plt.subplots(
#         1, len(split_types),
#         figsize=(4 * len(split_types), 5),
#         sharey=True
#     )
#     if len(split_types) == 1:
#         axes = [axes]

#     # positions = np.array([-1, 0, 1])   # centers
#     n_box = len(auc_cols)
#     positions = np.linspace(-1, 1, n_box)  # spread out
#     # width     = 1.0                    # full width -> boxes just touch
#     width     =  2/n_box            # narrower boxes

#     for ax, split in zip(axes, split_types):
#         df_sub = df_region[df_region["split_type"] == split]
#         data   = [df_sub[c].dropna().values for c in auc_cols]

#         bp = ax.boxplot(
#             data,
#             positions=positions,
#             widths=width,
#             patch_artist=True,
#             medianprops=dict(color="black")

#         )

#         for patch, color in zip(bp["boxes"], colors):
#             patch.set_facecolor(color)

#         ax.set_title(f"split_type = {split}")
#         ax.set_xlim(-1.6, 1.6)
#         ax.set_xticks([])

#     axes[0].set_ylabel("AUC")

#     # add title
#     fig.suptitle(f"Region: {region}", y=1.05, fontsize=16)

#     handles = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, auc_cols)]
#     fig.legend(handles, auc_cols, loc="upper center", ncol=1, bbox_to_anchor=(1.1, .5))

#     plt.tight_layout()
#     plt.show()


In [ ]:
results_df.sort_values(by = 'train_test_dist').plot(
    x="train_test_dist",
    y=["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA"],
    kind="line",
    marker="o",
    figsize=(8,5)
)


In [ ]:
df = results_df.sort_values('train_test_dist').copy()

# Apply rolling mean smoothing (window=3 is typical; adjust as needed)
df[['AUC_PO_only', 'AUC_PA_only', 'AUC_PO_plus_PA']] = (
    df[['AUC_PO_only', 'AUC_PA_only', 'AUC_PO_plus_PA']]
      .rolling(window=3, center=True)
      .mean()
)

df.plot(
    x="train_test_dist",
    y=["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA"],
    kind="line",
    marker="o",
    figsize=(8,5)
)


In [ ]:
# from statsmodels.nonparametric.smoothers_lowess import lowess

# df = results_df.sort_values('train_test_dist').copy()

# for col in ["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA"]:
#     smoothed = lowess(df[col], df['train_test_dist'], frac=0.3)
#     df[col] = smoothed[:,1]

# df.plot(
#     x="train_test_dist",
#     y=["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA"],
#     kind="line",
#     figsize=(8,5)
# )


In [ ]:
## probably shouldnt use this code as this is not valid when there are missing species

# import matplotlib.patches as mpatches
# import numpy as np

# # mean by region
# results_df = pd.read_csv('output/integration_geoplant/geoplant_partition_summary.csv')
# auc_cols    = ["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA",
#                "AUC_PO_plus_PA_ens", "AUC_PO_plus_PA_smooth"]
# results_df.groupby(['region', 'split_type'])[auc_cols].mean()

# auc_cols    = ["AUC_PO_only", "AUC_PA_only", "AUC_PO_plus_PA",
#                "AUC_PO_plus_PA_ens", "AUC_PO_plus_PA_smooth"]
# colors      = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
# split_types = ['closest', 'middle', 'farthest']

# fig, axes = plt.subplots(
#     1, len(split_types),
#     figsize=(4 * len(split_types), 5),
#     sharey=True
# )

# if len(split_types) == 1:
#     axes = [axes]

# n_box = len(auc_cols)
# positions = np.linspace(-1, 1, n_box)
# width     = 2/n_box  

# for ax, split in zip(axes, split_types):
#     df_sub = results_df[results_df["split_type"] == split]
#     data   = [df_sub[c].dropna().values for c in auc_cols]

#     # Boxplot
#     bp = ax.boxplot(
#         data,
#         positions=positions,
#         widths=width,
#         patch_artist=True,
#         medianprops=dict(color="black")
#     )

#     for patch, color in zip(bp["boxes"], colors):
#         patch.set_facecolor(color)

#     # === Mean Markers Only ===
#     means = [np.mean(d) for d in data]
#     print(means)
#     ax.scatter(
#         positions,
#         means,
#         s=80,
#         marker='o',
#         edgecolor='black',
#         facecolor='white',
#         zorder=3,
#         label="Mean"
#     )

#     ax.set_title(f"split_type = {split}")
#     ax.set_xlim(-1.6, 1.6)
#     ax.set_xticks([])

# axes[0].set_ylabel("AUC")

# # Legend
# handles = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, auc_cols)]
# handles.append(
#     mpatches.Patch(facecolor="white", edgecolor="black", label="Mean (○)")
# )

# fig.legend(handles, [*auc_cols, "Mean"], loc="upper center",
#            ncol=3, bbox_to_anchor=(0.5, 1.12))

# plt.tight_layout()
# plt.show()


## Checking full AUC

In [ ]:
results_df = pd.read_csv('output/integration_geoplant/geoplant_partition_summary.csv')
results_df_detailed = pd.read_csv('output/integration_geoplant/geoplant_partition_detailed_summary.csv')
auc_cols_sp = [f'{c}_sp' for c in auc_cols]

In [ ]:
results_df_detailed[['species', 'test_id'] + auc_cols_sp].head(200)

In [ ]:
results_df[auc_cols].mean()